# Módulo 3: Agentes de IA
- **Sesión:** 02 — Frameworks de agentes: LangGraph y protocolo MCP
- **Fecha:** 28/10/2026



---
## Fundamentos — El flujo como grafo



LangGraph representa una aplicación como un **grafo de estados**:

| Elemento | Qué es | En código |
|---|---|---|
| **Estado** | Diccionario compartido que viaja por el grafo | Una clase `TypedDict` |
| **Nodo** | Función que recibe el estado y devuelve las claves que actualiza | `grafo.add_node("nombre", funcion)` |
| **Arista** | Indica qué nodo se ejecuta después | `grafo.add_edge("a", "b")` |
| **Arista condicional** | Elige el siguiente nodo según el estado | `grafo.add_conditional_edges(...)` |

`START` y `END` son nodos especiales que marcan la entrada y la salida del grafo.


---
## Paso 0 — Configuración: modelo, datos y herramientas

### Concepto
`init_chat_model` crea el modelo a partir de `PREFIX_MODEL`. Las herramientas consultan los datos simulados de TiendaYa: son las mismas de la Sesión 1.

### Ejecutar

In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [2]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



In [ ]:
from langchain.tools import tool

PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}

POLITICAS = {
    "devoluciones": "15 días calendario desde la entrega; 7 días para productos electrónicos.",
    "envios": "Envío gratuito desde S/ 150; envío express en Lima por S/ 24.90.",
}


@tool
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado de un pedido de TiendaYa. Usar cuando el cliente menciona un número de pedido."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")


@tool
def consultar_politica(tema: str) -> str:
    """Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'envios'."""
    return POLITICAS.get(tema.lower(), "No existe una política sobre ese tema.")


SYSTEM_PROMPT = ("Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos. "
                 "Si la información no proviene de una herramienta, indica que no la tienes. Responde de forma breve.")

print(consultar_pedido.invoke({"numero_pedido": "48213"}))


def ver_grafo(app):
    """Muestra el diagrama del grafo como imagen; si no hay conexión, lo imprime en formato Mermaid."""
    try:
        from IPython.display import Image, display
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

---
## Paso 1 — Primer grafo: el encadenamiento como grafo

### Concepto
Se construye el workflow de encadenamiento de la Sesión 1 como un grafo de tres nodos en secuencia:

```
START ──▶ extraer ──▶ consultar ──▶ redactar ──▶ END
            │            │            │
         numero    estado_pedido   respuesta      ← claves que agrega cada nodo
```

La construcción sigue siempre tres pasos:
1. **Estado:** se declaran las claves que comparten los nodos.
2. **Nodos:** cada función recibe el estado completo y devuelve **solo** las claves que modifica; LangGraph las combina con el estado anterior.
3. **Grafo:** se agregan los nodos y las aristas, y se **compila**. `compile()` valida la estructura y devuelve un objeto ejecutable.

### Ejecutar: estado y nodos

In [ ]:
from typing import TypedDict


# 1. ESTADO: las claves que comparten los nodos
class Estado(TypedDict):
    consulta: str
    numero: str
    estado_pedido: str
    respuesta: str


# 2. NODOS: reciben el estado y devuelven un diccionario con lo que actualizan
def extraer(estado: Estado) -> dict:
    numero = llm.invoke(
        f"Extrae solo el número de pedido de este mensaje, sin otro texto: {estado['consulta']}"
    ).text.strip()
    return {"numero": numero}


def consultar(estado: Estado) -> dict:
    return {"estado_pedido": consultar_pedido.invoke({"numero_pedido": estado["numero"]})}


def redactar(estado: Estado) -> dict:
    texto = llm.invoke(
        "Responde de forma breve y cordial usando SOLO el dato indicado.\n"
        f"Consulta: {estado['consulta']}\nDato: {estado['estado_pedido']}"
    ).text
    return {"respuesta": texto}

### Ejecutar: grafo, compilación y diagrama

In [ ]:
from langgraph.graph import StateGraph, START, END

# 3. GRAFO
grafo = StateGraph(Estado)
grafo.add_node("extraer", extraer)
grafo.add_node("consultar", consultar)
grafo.add_node("redactar", redactar)

grafo.add_edge(START, "extraer")
grafo.add_edge("extraer", "consultar")
grafo.add_edge("consultar", "redactar")
grafo.add_edge("redactar", END)

app = grafo.compile()
ver_grafo(app)

### Ejecutar: paso a paso
`stream(..., stream_mode="updates")` devuelve, en cada paso, el nombre del nodo y las claves que actualizó. `invoke` devuelve solo el estado final.

In [ ]:
CONSULTA = "Hola, ¿dónde está mi pedido 48213?"

for paso in app.stream({"consulta": CONSULTA}, stream_mode="updates"):
    for nodo, cambios in paso.items():
        print(f"[{nodo}] {cambios}")

print("\nEstado final:")
final = app.invoke({"consulta": CONSULTA})
for clave, valor in final.items():
    print(f"  {clave}: {valor}")

---
## Paso 2 — Enrutamiento con aristas condicionales

### Concepto
Una **arista condicional** elige el siguiente nodo mediante una función que inspecciona el estado y devuelve el nombre de una rama. Un diccionario traduce cada valor posible al nodo correspondiente.

```
                       ┌─(pedido)───▶ rama_pedido ───┐
START ──▶ clasificar ──┼─(politica)─▶ rama_politica ─┼──▶ END
                       └─(otro)─────▶ derivar ───────┘
```

Es el workflow de enrutamiento de la Sesión 1: el **modelo** clasifica en un nodo y la **función de la arista**, escrita en el código, decide el camino. En el diagrama, las aristas condicionales aparecen con línea punteada.

### Ejecutar

In [ ]:
class EstadoRuta(TypedDict):
    consulta: str
    categoria: str
    respuesta: str


def clasificar(estado: EstadoRuta) -> dict:
    texto = llm.invoke(
        f"""
            Clasifica la consulta del cliente en una sola palabra: PEDIDO, POLITICA u OTRO.
            Consulta: {estado['consulta']}
        """
    ).text.strip().upper()
    return {"categoria": texto}


def rama_pedido(estado: EstadoRuta) -> dict:
    numero = llm.invoke(f"Extrae solo el número de pedido: {estado['consulta']}").text.strip()
    dato = consultar_pedido.invoke({"numero_pedido": numero})
    return {"respuesta": llm.invoke(f"Responde brevemente usando SOLO este dato: {dato}\nConsulta: {estado['consulta']}").text}


def rama_politica(estado: EstadoRuta) -> dict:
    tema = llm.invoke(f"¿Trata de 'devoluciones' o de 'envios'? Responde solo esa palabra: {estado['consulta']}").text.strip().lower()
    dato = consultar_politica.invoke({"tema": tema})
    return {"respuesta": llm.invoke(f"Responde brevemente usando SOLO esta política: {dato}\nConsulta: {estado['consulta']}").text}


def derivar(estado: EstadoRuta) -> dict:
    return {"respuesta": "Su consulta será atendida por un operador."}


# Función de la arista condicional: devuelve el nombre de la rama
def elegir_rama(estado: EstadoRuta) -> str:
    if "PEDIDO" in estado["categoria"]:
        return "pedido"
    if "POLITICA" in estado["categoria"]:
        return "politica"
    return "otro"


grafo = StateGraph(EstadoRuta)
grafo.add_node("clasificar", clasificar)
grafo.add_node("rama_pedido", rama_pedido)
grafo.add_node("rama_politica", rama_politica)
grafo.add_node("derivar", derivar)

grafo.add_edge(START, "clasificar")
grafo.add_conditional_edges(
    "clasificar",
    elegir_rama,
    {
        "pedido": "rama_pedido", 
        "politica": "rama_politica", 
        "otro": "derivar"
     },   # valor -> nodo
)
for nodo in ["rama_pedido", "rama_politica", "derivar"]:
    grafo.add_edge(nodo, END)

app_ruta = grafo.compile()
ver_grafo(app_ruta)

In [ ]:
for consulta in [
    "¿Dónde está mi pedido 50555?",
    "¿Cuántos días tengo para devolver un celular?",
    "¿Tienen tienda física en Trujillo?",
]:
    print(f"\nConsulta: {consulta}")
    for paso in app_ruta.stream({"consulta": consulta}, stream_mode="updates"):
        for nodo, cambios in paso.items():
            print(f"  [{nodo}] {list(cambios.values())[0][:110]}")

---
## Síntesis

| Paso | Concepto | Idea clave |
|---|---|---|
| 1 | Estado, nodo, arista | El estado viaja por el grafo; cada nodo devuelve solo lo que actualiza |
| 2 | Reductor | `add_messages` acumula; sin reductor, el valor se reemplaza |
| 3 | Arista condicional | Una función del código elige el siguiente nodo: enrutamiento |

| Patrón de la Sesión 1 | Forma del grafo |
|---|---|
| Encadenamiento | Nodos en secuencia con aristas fijas |
| Enrutamiento | Un nodo clasificador con aristas condicionales |
| Agente (ReAct) | Un ciclo modelo → herramientas → modelo (notebook siguiente) |

**Idea central:** un workflow es un grafo acíclico cuyo camino decide el código.

## Referencias
- LangChain. (2026). *LangGraph documentation*. https://docs.langchain.com/oss/python/langgraph/overview
- Oshin, M., & Campos, N. (2025). *Learning LangChain: Building AI and LLM applications with LangChain and LangGraph*. O'Reilly Media.
- Schluntz, E., & Zhang, B. (2024, 19 de diciembre). *Building effective agents*. Anthropic. https://www.anthropic.com/research/building-effective-agents